In [1]:
import sys
import pandas as pd
import numpy as np
from glob import glob
from sklearn.metrics import accuracy_score, f1_score, classification_report

sys.path.append("../Config")
from config import DATASETS, INPUT_SETTINGS, LABEL_SETTINGS

In [2]:
# 100 random subsamples of 500 test memes (without replacement).
# Note: 500 memes is 1/6 of FHM's test set but half of MAMI's, so the stds are not comparable across datasets;
# set REPLACE = True and SAMPLE_SIZE = None for a standard bootstrap (n memes with replacement) on both datasets.
SAMPLE_SIZE = 500
REPLACE = False

def bootstrap_performance(y_true, y_pred, sample_size=500, n_samples=100, replace=False):
    assert len(y_true) == len(y_pred), "Both lists must be of same length"
    sample_size = len(y_true) if sample_size is None else min(sample_size, len(y_true))
    accuracies = []
    f1s = []

    for seed_val in range(n_samples):
        # Sample indices
        np.random.seed(seed_val)
        indices = np.random.choice(len(y_true), size=sample_size, replace=replace)
        sample_true = [y_true[i] for i in indices]
        sample_pred = [y_pred[i] for i in indices]
        acc = accuracy_score(sample_true, sample_pred)
        accuracies.append(acc)
        f1 = f1_score(sample_true, sample_pred, average='weighted')
        f1s.append(f1)

    return np.mean(accuracies), np.std(accuracies), np.mean(f1s), np.std(f1s)

def analyze_results(tru, pred):
    accuracy = accuracy_score(tru, pred)
    f1 = f1_score(tru, pred, average='weighted')
    report = classification_report(tru, pred, digits=6, zero_division=0)

    print(f"Overall Accuracy: {100*accuracy:.2f}")
    print(f"Overall w-F1 Score: {100*f1:.2f}")
    print("Overall Classification Report:")
    print(report)

    # Mean and standard deviation of the predictions performance scores
    mean_acc, std_acc, mean_f1, std_f1 = bootstrap_performance(tru, pred, SAMPLE_SIZE, replace=REPLACE)
    print(f"Bootstrapped Accuracy: {100*mean_acc:.2f} ± {100*std_acc:.2f}")
    print(f"Bootstrapped w-F1 Score: {100*mean_f1:.2f} ± {100*std_f1:.2f}")
    return mean_acc, std_acc, mean_f1, std_f1

In [3]:
# (category, display name, size) of each model, in table order
MODELS = {
    "albert-base-v2": ("Unimodal (text)", "ALBERT", "12M"),
    "bert-base-uncased": ("Unimodal (text)", "BERT", "112M"),
    "roberta-base": ("Unimodal (text)", "RoBERTa", "125M"),
    "vit-base-patch16-224": ("Unimodal (vision)", "ViT", "86M"),
    "beit-base-patch16-224-pt22k-ft22k": ("Unimodal (vision)", "BEiT", "86M"),
    "deit-base-patch16-224": ("Unimodal (vision)", "DeiT", "86M"),
    "clip-vit-base-patch32": ("Multimodal", "CLIP", "150M"),
    "SmolVLM-Instruct": ("Multimodal", "SmolVLM", "2B"),
    "Qwen2.5-VL-3B-Instruct": ("Multimodal", "Qwen2.5-VL", "3B"),
    "Qwen2-VL-7B-Instruct": ("Multimodal", "Qwen2-VL", "7B"),
}
TEXT_MODELS = ["albert-base-v2", "bert-base-uncased", "roberta-base"]
VLMS = ["SmolVLM-Instruct", "Qwen2.5-VL-3B-Instruct", "Qwen2-VL-7B-Instruct"]

# Outputs/<dataset>/<model>_<setting>.csv (fine-tuned) or <model>_ZS_<setting>.csv (zero-shot)
def parse_file(f):
    stem = f.split("/")[-1][:-4]
    for setting in INPUT_SETTINGS + LABEL_SETTINGS:
        if stem.endswith("_" + setting):
            model = stem[: -len(setting) - 1]
            if model.endswith("_ZS"):
                return model[:-3], "ZS", setting
            return model, "FT", setting
    return None

In [4]:
rows = []
for dataset in DATASETS:
    files = glob(f"Outputs/{dataset}/*.csv")
    files.sort()

    for f in files:
        parsed = parse_file(f)
        if parsed is None:
            continue
        model, learning, setting = parsed
        print(f"Analyzing file: {f}")
        df = pd.read_csv(f)
        tru = df['true_label'].tolist()
        pred = df['predicted_label'].tolist()
        if (df['predicted_label'] == 2).any():
            print(f"Unclear outputs (counted as wrong): {(df['predicted_label'] == 2).sum()}")

        # Analyze the results
        mean_acc, std_acc, mean_f1, std_f1 = analyze_results(tru, pred)
        rows.append({"dataset": dataset, "model": model, "learning": learning, "setting": setting,
                     "acc": 100*mean_acc, "acc_std": 100*std_acc, "f1": 100*mean_f1, "f1_std": 100*std_f1})
        print("\n" + "="*50 + "\n")

results = pd.DataFrame(rows)
results.to_csv("Outputs/results_all.csv", index=False)

Analyzing file: Outputs/FHM/Qwen2-VL-7B-Instruct_ZS_category_binary.csv
Overall Accuracy: 67.67
Overall w-F1 Score: 66.57
Overall Classification Report:
              precision    recall  f1-score   support

           0   0.688995  0.818182  0.748052      1760
           1   0.648352  0.475806  0.548837      1240

    accuracy                       0.676667      3000
   macro avg   0.668673  0.646994  0.648445      3000
weighted avg   0.672196  0.676667  0.665710      3000

Bootstrapped Accuracy: 67.64 ± 1.73
Bootstrapped w-F1 Score: 66.50 ± 1.83


Analyzing file: Outputs/FHM/Qwen2-VL-7B-Instruct_ZS_category_scale.csv
Overall Accuracy: 67.80
Overall w-F1 Score: 66.14
Overall Classification Report:
              precision    recall  f1-score   support

           0   0.680783  0.849432  0.755814      1760
           1   0.670398  0.434677  0.527397      1240

    accuracy                       0.678000      3000
   macro avg   0.675591  0.642055  0.641606      3000
weighted avg   0.676

### Result tables
Cells are `mean ± std` over the bootstrap samples; `-` means the run is missing. Tables are also saved to `Outputs/tables/`.

In [5]:
def cell(dataset, model, learning, setting, metric):
    r = results[(results.dataset == dataset) & (results.model == model) & (results.learning == learning) & (results.setting == setting)] if len(results) else results
    if len(r) == 0:
        return "-"
    return f"{r.iloc[0][metric]:.2f} ± {r.iloc[0][metric + '_std']:.2f}"

# Labels: prompt (simple/category) x label (binary/scaled), fine-tuning and zero-shot prompting
def labels_table(dataset):
    out = []
    for learning, models in [("FT", TEXT_MODELS + VLMS), ("ZS", VLMS)]:
        for model in models:
            category, name, size = MODELS[model]
            for prompt in ["simple", "category"]:
                row = {"Category": f"{category} ({'fine-tuning' if learning == 'FT' else 'prompting'})", "Model": name, "Size": size, "Prompt": prompt.title()}
                for col, label in [("Original (Binary)", "binary"), ("Augmented (Scaled)", "scale")]:
                    row[f"{col} Accuracy"] = cell(dataset, model, learning, f"{prompt}_{label}", "acc")
                    row[f"{col} F1-score"] = cell(dataset, model, learning, f"{prompt}_{label}", "f1")
                out.append(row)
    return pd.DataFrame(out)

# Inputs: original vs extended training data
def inputs_table(dataset):
    out = []
    for model, (category, name, size) in MODELS.items():
        row = {"Category": f"{category} (fine-tuning)", "Model": name, "Size": size}
        for col, setting in [("Original", "orig"), ("Augmented (Extended)", "inputs")]:
            row[f"{col} Accuracy"] = cell(dataset, model, "FT", setting, "acc")
            row[f"{col} F1-score"] = cell(dataset, model, "FT", setting, "f1")
        out.append(row)
    return pd.DataFrame(out)

# Zero-shot VLMs with the Yes/No prompt
def zeroshot_table(dataset):
    out = []
    for model in VLMS:
        category, name, size = MODELS[model]
        out.append({"Category": f"{category} (zero-shot ICL)", "Model": name, "Size": size,
                    "Accuracy": cell(dataset, model, "ZS", "orig", "acc"), "F1-score": cell(dataset, model, "ZS", "orig", "f1")})
    return pd.DataFrame(out)

In [6]:
import os
os.makedirs("Outputs/tables", exist_ok=True)

for dataset in DATASETS:
    for name, table in [("labels", labels_table), ("inputs", inputs_table), ("zeroshot", zeroshot_table)]:
        t = table(dataset)
        t.to_csv(f"Outputs/tables/{dataset}_{name}.csv", index=False)
        print(f"##### {dataset}: {name} #####")
        display(t)

##### FHM: labels #####


,Category,Model,Size,Prompt,Original (Binary) Accuracy,Original (Binary) F1-score,Augmented (Scaled) Accuracy,Augmented (Scaled) F1-score
0,Unimodal (text) (fine-tuning),ALBERT,12M,Simple,61.74 ± 1.79,60.30 ± 1.87,60.97 ± 2.04,53.20 ± 2.59
1,Unimodal (text) (fine-tuning),ALBERT,12M,Category,62.91 ± 1.82,59.77 ± 2.01,61.01 ± 1.83,51.11 ± 2.30
2,Unimodal (text) (fine-tuning),BERT,112M,Simple,62.10 ± 1.84,59.40 ± 2.04,60.48 ± 1.86,51.77 ± 2.37
3,Unimodal (text) (fine-tuning),BERT,112M,Category,62.16 ± 1.93,57.85 ± 2.24,61.32 ± 1.96,51.90 ± 2.58
4,Unimodal (text) (fine-tuning),RoBERTa,125M,Simple,62.44 ± 1.93,59.51 ± 2.15,61.26 ± 2.01,51.62 ± 2.56
5,Unimodal (text) (fine-tuning),RoBERTa,125M,Category,61.83 ± 2.06,58.74 ± 2.28,61.15 ± 2.00,52.51 ± 2.56
6,Multimodal (fine-tuning),SmolVLM,2B,Simple,68.87 ± 1.70,67.34 ± 1.86,64.24 ± 1.91,57.66 ± 2.41
7,Multimodal (fine-tuning),SmolVLM,2B,Category,68.23 ± 1.89,66.28 ± 2.07,62.56 ± 1.97,54.54 ± 2.52
8,Multimodal (fine-tuning),Qwen2.5-VL,3B,Simple,75.41 ± 1.72,75.05 ± 1.75,64.82 ± 1.93,58.44 ± 2.39
9,Multimodal (fine-tuning),Qwen2.5-VL,3B,Category,75.11 ± 1.71,74.48 ± 1.80,65.90 ± 1.79,61.22 ± 2.16


##### FHM: inputs #####


,Category,Model,Size,Original Accuracy,Original F1-score,Augmented (Extended) Accuracy,Augmented (Extended) F1-score
0,Unimodal (text) (fine-tuning),ALBERT,12M,61.44 ± 1.94,60.14 ± 1.98,61.57 ± 1.98,59.34 ± 2.19
1,Unimodal (text) (fine-tuning),BERT,112M,62.19 ± 1.78,58.38 ± 2.13,62.30 ± 1.68,58.91 ± 1.92
2,Unimodal (text) (fine-tuning),RoBERTa,125M,62.11 ± 1.87,59.92 ± 1.99,62.12 ± 1.86,58.74 ± 2.12
3,Unimodal (vision) (fine-tuning),ViT,86M,58.66 ± 1.84,52.03 ± 2.19,59.18 ± 2.07,54.52 ± 2.32
4,Unimodal (vision) (fine-tuning),BEiT,86M,58.86 ± 2.09,57.72 ± 2.15,59.06 ± 2.11,57.69 ± 2.23
5,Unimodal (vision) (fine-tuning),DeiT,86M,57.08 ± 1.89,54.58 ± 2.08,57.39 ± 2.12,54.70 ± 2.29
6,Multimodal (fine-tuning),CLIP,150M,64.38 ± 1.91,60.03 ± 2.25,66.63 ± 1.73,64.64 ± 1.91
7,Multimodal (fine-tuning),SmolVLM,2B,69.79 ± 1.86,68.29 ± 2.02,68.71 ± 1.69,66.26 ± 1.89
8,Multimodal (fine-tuning),Qwen2.5-VL,3B,75.27 ± 1.57,74.71 ± 1.63,74.67 ± 1.67,73.59 ± 1.83
9,Multimodal (fine-tuning),Qwen2-VL,7B,80.54 ± 1.56,80.29 ± 1.61,81.65 ± 1.62,81.47 ± 1.67


##### FHM: zeroshot #####


,Category,Model,Size,Accuracy,F1-score
0,Multimodal (zero-shot ICL),SmolVLM,2B,58.68 ± 1.84,43.59 ± 2.23
1,Multimodal (zero-shot ICL),Qwen2.5-VL,3B,63.55 ± 1.76,63.82 ± 1.74
2,Multimodal (zero-shot ICL),Qwen2-VL,7B,69.82 ± 1.77,69.13 ± 1.85


... (2 outputs left out)

##### MAMI: inputs #####


,Category,Model,Size,Original Accuracy,Original F1-score,Augmented (Extended) Accuracy,Augmented (Extended) F1-score
0,Unimodal (text) (fine-tuning),ALBERT,12M,63.19 ± 1.46,62.01 ± 1.52,63.23 ± 1.52,62.32 ± 1.56
1,Unimodal (text) (fine-tuning),BERT,112M,63.43 ± 1.56,62.38 ± 1.60,61.92 ± 1.51,61.03 ± 1.54
2,Unimodal (text) (fine-tuning),RoBERTa,125M,63.24 ± 1.57,62.59 ± 1.62,62.62 ± 1.45,62.03 ± 1.47
3,Unimodal (vision) (fine-tuning),ViT,86M,65.14 ± 1.42,63.81 ± 1.51,64.51 ± 1.43,63.09 ± 1.50
4,Unimodal (vision) (fine-tuning),BEiT,86M,64.49 ± 1.73,64.12 ± 1.75,65.29 ± 1.46,64.38 ± 1.49
5,Unimodal (vision) (fine-tuning),DeiT,86M,62.23 ± 1.42,60.64 ± 1.49,62.21 ± 1.47,60.74 ± 1.51
6,Multimodal (fine-tuning),CLIP,150M,66.99 ± 1.26,65.40 ± 1.34,68.86 ± 1.29,67.56 ± 1.38
7,Multimodal (fine-tuning),SmolVLM,2B,70.52 ± 1.19,69.27 ± 1.25,71.46 ± 1.24,70.59 ± 1.31
8,Multimodal (fine-tuning),Qwen2.5-VL,3B,76.03 ± 1.28,75.18 ± 1.36,76.73 ± 1.39,75.98 ± 1.46
9,Multimodal (fine-tuning),Qwen2-VL,7B,80.26 ± 1.32,79.87 ± 1.38,80.19 ± 1.22,79.76 ± 1.26


##### MAMI: zeroshot #####


,Category,Model,Size,Accuracy,F1-score
0,Multimodal (zero-shot ICL),SmolVLM,2B,50.25 ± 1.65,34.36 ± 1.89
1,Multimodal (zero-shot ICL),Qwen2.5-VL,3B,68.47 ± 1.51,68.27 ± 1.53
2,Multimodal (zero-shot ICL),Qwen2-VL,7B,62.39 ± 1.58,57.80 ± 1.82
